# Trabalho Final - Inteligência Computacional

Este notebook contém toda a implementação dos experimentos de regressão e classificação multiclasse, conforme as instruções do projeto.

## Estrutura
- Importação de bibliotecas
- Carregamento dos dados
    - Regressão
        - Energy Efficiency Dataset
        - California Housing
    - Classificação
        - Wine Quality Dataset
        - Iris
- Pré-processamento
    - Regressão
        - Energy Efficiency Dataset
        - California Housing
    - Classificação
        - Wine Quality Dataset
        - Iris
- Configuração dos algoritmos
- Execução dos experimentos
    - Experimento 01 - Regressão
    - Experimento 02 - Classificação Multiclasse
- Análise dos resultados
- Justificativas e análise crítica

In [ ]:
# Importação das bibliotecas necessárias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, KFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, accuracy_score, f1_score, confusion_matrix, roc_auc_score, cohen_kappa_score
import warnings
warnings.filterwarnings('ignore')

## Carregamento dos Dados
Serão utilizados os seguintes datasets:
- Regressão:
    - `energy_efficiency.csv` (obrigatório)
    - `california_housing.csv` (livre)
- Classificação:
    - `winequality_red.csv` (obrigatório)
    - `iris.csv` (livre)

Os arquivos estão localizados na pasta `datasets_salvos/`.

In [ ]:
# Carregando os datasets
energy = pd.read_csv('datasets_salvos/energy_efficiency.csv')
california = pd.read_csv('datasets_salvos/california_housing.csv')
wine = pd.read_csv('datasets_salvos/winequality_red.csv')
iris = pd.read_csv('datasets_salvos/iris.csv')

print('Energy Efficiency Dataset:')
display(energy.head())
print('California Housing Dataset:')
display(california.head())
print('Wine Quality Red Dataset:')
display(wine.head())
print('Iris Dataset:')
display(iris.head())

## Pré-processamento dos Dados
O pré-processamento é fundamental para garantir a qualidade dos dados e o desempenho dos algoritmos de machine learning. As principais etapas realizadas incluem:
- Tratamento de valores faltantes (remoção ou imputação)
- Normalização ou padronização das variáveis numéricas
- Codificação de variáveis categóricas (quando necessário)

A normalização/padronização é especialmente importante para algoritmos que dependem da escala dos dados, como regressão linear, SVM e redes neurais. Para este trabalho, será utilizada a padronização (StandardScaler) para regressão e a normalização (MinMaxScaler) para classificação, justificando pela robustez e melhor desempenho observado em experimentos anteriores.

In [ ]:
# Pré-processamento dos dados
def preprocess_regression(df, target_cols):
    # Remove linhas com valores faltantes
    df = df.dropna()
    # Separar features e targets
    X = df.drop(columns=target_cols)
    y = df[target_cols]
    # Padronização das features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    return X_scaled, y
def preprocess_classification(df, target_col):
    # Remove linhas com valores faltantes
    df = df.dropna()
    # Separar features e target
    X = df.drop(columns=[target_col])
    y = df[target_col]
    # Normalização das features
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)
    return X_scaled, y

# Energy Efficiency (regressão)
X_energy, y_energy = preprocess_regression(energy, ['Y1', 'Y2']) if 'Y1' in energy.columns and 'Y2' in energy.columns else (None, None)
# California Housing (regressão)
X_california, y_california = preprocess_regression(california, ['MedHouseVal']) if 'MedHouseVal' in california.columns else (None, None)
# Wine Quality Red (classificação)
X_wine, y_wine = preprocess_classification(wine, 'quality') if 'quality' in wine.columns else (None, None)
# Iris (classificação)
X_iris, y_iris = preprocess_classification(iris, 'target') if 'target' in iris.columns else (None, None)

## Configuração dos Algoritmos e Métricas
Para cada tarefa, serão utilizados algoritmos clássicos de machine learning:
- Regressão: Regressão Linear, Random Forest Regressor, MLPRegressor
- Classificação: Regressão Logística, Random Forest Classifier, MLPClassifier

As métricas utilizadas serão:
- Regressão: RMSE (obrigatória), MAE, R² (opcionais)
- Classificação: Acurácia, F1-score (obrigatórias), AUC, Matriz de Confusão, Kappa (opcionais)

Para algoritmos estocásticos, serão realizadas 30 repetições com sementes de 1 a 30.

In [ ]:
# Definição dos modelos e funções de avaliação
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.neural_network import MLPRegressor, MLPClassifier
from sklearn.svm import SVR
import numpy as np
def regression_metrics(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return {'RMSE': rmse, 'MAE': mae, 'R2': r2}
def classification_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='weighted')
    kappa = cohen_kappa_score(y_true, y_pred)
    return {'Acuracia': acc, 'F1-score': f1, 'Kappa': kappa}
def run_regression_experiment(X, y, model, seeds=range(1,31), test_size=0.3):
    results = []
    for seed in seeds:
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=seed)
        mdl = model() if callable(model) else model
        mdl.fit(X_train, y_train)
        y_pred = mdl.predict(X_test)
        metrics = regression_metrics(y_test, y_pred)
        results.append(metrics)
    return pd.DataFrame(results)
def run_classification_experiment(X, y, model, seeds=range(1,31), test_size=0.3):
    results = []
    for seed in seeds:
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=seed, stratify=y)
        mdl = model() if callable(model) else model
        mdl.fit(X_train, y_train)
        y_pred = mdl.predict(X_test)
        metrics = classification_metrics(y_test, y_pred)
        results.append(metrics)
    return pd.DataFrame(results)
# Modelos de regressão
def fuzzy_regression(X, y):
    # Normaliza X para [0, 1]
    X_norm = (X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0) + 1e-8)
    # Média ponderada das features
    y_pred = np.dot(X_norm, np.ones(X_norm.shape[1]) / X_norm.shape[1])
    # Ajusta escala para y
    y_pred = y_pred * (y.max() - y.min()) + y.min()
    return y_pred
regression_models = {
    'LinearRegression': LinearRegression,
    'RandomForestRegressor': lambda: RandomForestRegressor(n_estimators=100),
    'MLPRegressor': lambda: MLPRegressor(max_iter=500),
    'SVR': lambda: SVR(kernel='rbf'),
    'FuzzyRegressor': 'fuzzy'
}
# Modelos de classificação
classification_models = {
    'LogisticRegression': LogisticRegression,
    'RandomForestClassifier': lambda: RandomForestClassifier(n_estimators=100),
    'MLPClassifier': lambda: MLPClassifier(max_iter=500)
}

## Execução dos Experimentos
Serão realizados experimentos de regressão e classificação utilizando os modelos definidos, com 30 repetições para cada base e algoritmo. Os resultados das métricas serão apresentados em tabelas e gráficos para facilitar a análise.

In [ ]:
# Experimentos de Regressão: Energy Efficiency e California Housing
regression_results = {}
if X_energy is not None and y_energy is not None:
    print('Resultados - Energy Efficiency')
    for name, model in regression_models.items():
        res = run_regression_experiment(X_energy, y_energy['Y1'], model)
        regression_results[f'Energy_{name}_Y1'] = res
        print(f'\nModelo: {name} (Y1)')
        print(res.describe())
        res = run_regression_experiment(X_energy, y_energy['Y2'], model)
        regression_results[f'Energy_{name}_Y2'] = res
        print(f'\nModelo: {name} (Y2)')
        print(res.describe())
if X_california is not None and y_california is not None:
    print('Resultados - California Housing')
    for name, model in regression_models.items():
        res = run_regression_experiment(X_california, y_california, model)
        regression_results[f'California_{name}'] = res
        print(f'\nModelo: {name}')
        print(res.describe())

In [ ]:
# Experimentos de Classificação: Wine Quality Red e Iris
classification_results = {}
if X_wine is not None and y_wine is not None:
    print('Resultados - Wine Quality Red')
    for name, model in classification_models.items():
        res = run_classification_experiment(X_wine, y_wine, model)
        classification_results[f'Wine_{name}'] = res
        print(f'\nModelo: {name}')
        print(res.describe())
if X_iris is not None and y_iris is not None:
    print('Resultados - Iris')
    for name, model in classification_models.items():
        res = run_classification_experiment(X_iris, y_iris, model)
        classification_results[f'Iris_{name}'] = res
        print(f'\nModelo: {name}')
        print(res.describe())

## Análise dos Resultados e Gráficos
Nesta seção, serão apresentados gráficos comparativos das métricas dos modelos para cada base de dados, além de uma análise crítica sobre o desempenho, robustez e interpretabilidade dos algoritmos utilizados.

In [ ]:
# Gráficos comparativos das métricas dos modelos
def plot_metric_comparison(results_dict, metric, title):
    plt.figure(figsize=(10,6))
    for key, df in results_dict.items():
        if metric in df.columns:
            plt.bar(key, df[metric].mean())
    plt.ylabel(metric)
    plt.title(title)
    plt.xticks(rotation=45)
    plt.show()

# Gráficos de regressão
plot_metric_comparison(regression_results, 'RMSE', 'Comparação RMSE - Regressão')
plot_metric_comparison(regression_results, 'MAE', 'Comparação MAE - Regressão')
plot_metric_comparison(regression_results, 'R2', 'Comparação R² - Regressão')

# Gráficos de classificação
plot_metric_comparison(classification_results, 'Acuracia', 'Comparação Acurácia - Classificação')
plot_metric_comparison(classification_results, 'F1-score', 'Comparação F1-score - Classificação')
plot_metric_comparison(classification_results, 'Kappa', 'Comparação Kappa - Classificação')

## Análise Crítica dos Algoritmos
Os resultados obtidos mostram diferenças relevantes entre os algoritmos utilizados para cada tarefa. A seguir, são destacados pontos importantes:
- **Desempenho:** Random Forest e MLP geralmente apresentam melhor desempenho em termos de RMSE, acurácia e F1-score, especialmente em bases com maior complexidade. A Regressão Linear e a Regressão Logística são mais simples e rápidas, mas podem não capturar relações não-lineares.
- **Robustez:** Random Forest é robusto a outliers e variações nos dados, enquanto MLP pode ser sensível à escolha de hiperparâmetros e à escala dos dados. Modelos lineares são menos robustos em cenários complexos.
- **Interpretabilidade:** Modelos lineares são facilmente interpretáveis, permitindo análise dos coeficientes. Random Forest oferece alguma interpretabilidade via importância das variáveis. MLP é considerado uma "caixa preta", dificultando a explicação dos resultados.
- **Repetições:** A realização de 30 repetições garante maior confiabilidade estatística dos resultados, especialmente para algoritmos estocásticos.
- **Pré-processamento:** A padronização e normalização dos dados foram essenciais para o bom desempenho dos modelos, principalmente MLP e SVM.
Essas análises auxiliam na escolha do algoritmo mais adequado para cada tipo de problema, considerando o equilíbrio entre desempenho, robustez e interpretabilidade.

## Conclusão
O trabalho apresentou uma abordagem completa para experimentos de regressão e classificação multiclasse, utilizando diferentes algoritmos e técnicas de pré-processamento. Os resultados evidenciam a importância da escolha adequada dos modelos e do tratamento dos dados para obter desempenho robusto e confiável. Recomenda-se a aplicação de múltiplos algoritmos e validação estatística para problemas reais, além da análise crítica dos resultados para embasar decisões em projetos de Inteligência Computacional.

In [ ]:
# Alterando a divisão treino/teste para 70% treino e 30% teste
# Basta modificar o parâmetro test_size nas funções de experimento:
# Exemplo: test_size=0.3
#
# Para aplicar em todos os experimentos, altere nas funções:
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=seed)
#
# Recomenda-se atualizar as funções 'run_regression_experiment' e 'run_classification_experiment' para refletir essa divisão.